<p><img src="tools/demo_ui/static/fabric-logo.png" alt="FABRIC" height="40" style="display:inline;margin-right:24px"><img src="tools/demo_ui/static/amlight-logo.jpg" alt="AmLight" height="54" style="display:inline"></p>

# Globus between Cape Town and FABRIC: data on the research path

**The story in one line:** a Globus transfer between SANReN in Cape Town and FABRIC moves its bytes over a
dedicated research VLAN across the Atlantic (AmLight / RNP), not the public internet. Nobody has to choose that
path when starting the transfer. It follows from how the endpoints are set up.

> **Talking points (presenter)**
> 1. *Two addresses per endpoint.* Each FABRIC DTN has a public address for Globus control and a private address on the research VLAN for data.
> 2. *The data address is what the endpoint advertises.* Globus Connect Server's `--data-interface` is the VLAN address. Globus passes it to Cape Town, and Cape Town connects to it, so the bytes take the VLAN.
> 3. *Where you click doesn't matter.* The CLI, the web app and this notebook only ask the Globus cloud to move data. The data never passes through them, so all three take the same path. Section 5c shows this live.
> 4. *Proof from the wire, not a diagram.* During each transfer we read the DTN's own TCP sockets and chart how many bytes each network interface carried.
> 5. *FABRIC as glue.* A facility port bridges a FABRIC VM straight onto the AmLight VLAN, with no routing and no tunnels, and the same weave builds it at FIU and CIEN.

| address on each DTN | used for | who connects to it |
|---|---|---|
| public (AmLight-Layer3-FIU / FABnetv4Ext) | registration, Globus control (TCP 443) | the Globus cloud service |
| data NIC (L2 facility-port VLAN) | **GridFTP data** (TCP 50000-51000), GCS `--data-interface` | the Cape Town DTN |

**Before the demo:**
- The slice is up (`ACTION=status ./weave.sh <prefix>` shows Cape Town reachable).
- The Globus identity is in `GLOBUS_USERS` and has R/W on the Cape Town collection.
- `CAPE_SRC_PATH` points at a dataset large enough to run for at least 30 s.

## 1. What the weave built

In [ ]:
import importlib.util, subprocess, sys
# LoomAI's Jupyter may not have these yet; install them into this kernel once.
_need = [p for m, p in (("globus_sdk", "globus-sdk>=3.30,<4"), ("plotly", "plotly")) if not importlib.util.find_spec(m)]
if _need:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_need])

import json, os, re, time
from datetime import datetime, timezone
import matplotlib.pyplot as plt
from IPython.display import display
import demo_viz
from fabric_env import get_fablib, load_legs

CAPE_COLLECTION = "35df6f87-df63-11f1-977e-02ffe3faed53"   # SANReN GCP - Cape Town SC26 (cryoem-without-borders)
CAPE_SRC_PATH   = "/"              # what to pull from Cape Town (file or directory in the guest collection)
CAPE_DST_DIR    = "/from-fabric/"  # where pushes land on the Cape Town collection (needs R/W there)
PUSH_SIZE_GB    = 10               # size of the generated file for the DTN -> Cape Town run

built = load_legs()
legs = {s: l for s, l in built["legs"].items() if "error" not in l}
for s, l in built["legs"].items():
    if "error" in l:
        print(f"{s}: FAILED in the weave - {l['error'].splitlines()[0]}")
    else:
        print(f"{s}: data {l['data_addr']} on {l.get('l2_fp')} VLAN {l.get('l2_vlan')} (MTU {l['mtu']}) -> {l['peer_ip']}; "
              f"public {l.get('public_ip', '-')} / {l.get('public_ip6') or '-'}; Globus {l.get('globus')}; "
              f"collection {l.get('collection_id', '-')}")

fablib = get_fablib()
def node(site):
    return fablib.get_slice(name=legs[site]["slice"]).get_node(name="dtn")

### The demo web app

Each DTN runs a small web app (the same story as this notebook, for the booth screen): path diagram, live
per-path chart, the VLAN/public switch, and transfer buttons. The weave installs it; it listens on
`127.0.0.1:8080` on the DTN only, so you reach it through a tunnel:

- **LoomAI:** Tunnels > the slice below, node `dtn`, port `8080`, then open the tunnel URL.
- **Laptop:** run the `ssh -N -L 8080:...` command printed below, then open http://localhost:8080/.

Set `INSTALL_APP = True` to install it (or update it after changing `tools/demo_ui/`).

In [ ]:
import provision

INSTALL_APP = False   # True = install / update the app on each DTN first

for site, l in legs.items():
    lines = provision.install_app(fablib, l) if INSTALL_APP else provision.app_access(l)
    print(f"--- {site} (slice {l['slice']})")
    for line in lines:
        print("  " + line)

## 2. The path

The green line is the Cape Town VLAN: SANReN to Sangano, across the South Atlantic to Fortaleza, up to Miami on
AmLight, and into FABRIC at FIU through a layer-2 facility port. Hover over a DTN to see its addresses. Globus
control traffic doesn't use this line; it reaches each DTN on its public address.

In [ ]:
demo_viz.path_map(fablib, legs).show()

### On the AmLight and FABRIC maps

The same path on the two networks' own maps. On AmLight, the Cape Town VLAN runs from SANReN through Sangano and
Fortaleza to Miami (the route drawn is approximate). On FABRIC, it enters at FIU/AMPATH through the facility port
and lands on the DTN's data NIC.

In [ ]:
demo_viz.network_maps(", ".join(legs) or "FIU");

## 3. Is the path up?

`ip route get` must show the **data NIC**, not `eth0` (management) or the public uplink. Cape Town only answers
once the operators have mapped the FABRIC VLAN to the Cape Town VLAN on the switch.

In [ ]:
for site, l in legs.items():
    out, _ = node(site).execute(
        f"ip route get {l['peer_ip']} | head -1; "
        f"ping -c 5 -q {l['peer_ip']} | tail -2; "
        f"ping -c 2 -W 2 -M do -s {l['mtu'] - 28} {l['peer_ip']} >/dev/null && echo 'MTU {l['mtu']} passes' "
        f"|| echo 'MTU {l['mtu']} does NOT pass'", quiet=True)
    print(f"--- {site} ({l['data_dev']} {l['data_addr']})\n{out}")

### iperf3 baseline (optional)

This needs `iperf3 -s` running on the Cape Town DTN (TCP 5201 is open). It shows what the raw path can do, for
comparison with the Globus rate later.

In [ ]:
for site, l in legs.items():
    out, _ = node(site).execute(
        f"timeout 40 iperf3 -c {l['peer_ip']} -B {l['data_ip']} -P 8 -t 20 -f g 2>&1 | tail -4", quiet=True)
    print(f"--- {site} -> Cape Town\n{out}")

## 4. What each endpoint tells Globus

`node list` shows the public address the endpoint is registered on, and the **data interface** it hands to the
other side. That one setting is what puts the data on the research path.

In [ ]:
for site, l in legs.items():
    if l.get("globus") != "ready":
        print(f"{site}: Globus not registered ({l.get('globus')})"); continue
    out, _ = node(site).execute(
        "sudo bash -c '. /etc/globus-dtn/creds.env && export GCS_CLI_CLIENT_ID GCS_CLI_CLIENT_SECRET "
        "GCS_CLI_ENDPOINT_ID=$(cat /etc/globus-dtn/endpoint-id.txt) && globus-connect-server node list'",
        quiet=True)
    print(f"--- {site}: public {l['public_ip']}, data {l['data_ip']}\n{out}")

### 4b. Switch the data path (VLAN or public)

The same switch as the **Data via VLAN / Data via public** buttons in the web app. `public` makes the endpoint
advertise its public address for data, which gives a "before" picture (bytes on the internet) and lets transfers
run before the VLAN is mapped. Set `DATA_PATH_MODE` to `show`, `vlan` or `public`; the change applies to the next
transfer.

In [ ]:
DATA_PATH_MODE = "show"   # "show" | "vlan" | "public"

for site, l in legs.items():
    if l.get("globus") != "ready":
        print(f"{site}: Globus not registered ({l.get('globus')})"); continue
    out, _ = node(site).execute(f"sudo /usr/local/sbin/globus-dtn-data-path {DATA_PATH_MODE} 2>&1 | tail -3", quiet=True)
    print(f"--- {site}\n{out}")

### Sign in to Globus

Open the link, sign in with an identity that is in `GLOBUS_USERS` and has access to the Cape Town collection, and
paste the code back. `GLOBUS_NATIVE_CLIENT_ID` defaults to the Globus SDK tutorial client.

In [ ]:
import globus_sdk

CLIENT_ID = os.environ.get("GLOBUS_NATIVE_CLIENT_ID", "61338d24-54d5-408f-a10d-66c06b59f6d2")
mapped = [l["collection_id"] for l in legs.values() if l.get("collection_id")]
# GCS 5 mapped collections need a data_access consent each; the Cape Town GCP collection does not.
scope = "urn:globus:auth:scope:transfer.api.globus.org:all"
if mapped:
    scope += "[" + " ".join(f"*https://auth.globus.org/scopes/{c}/data_access" for c in mapped) + "]"

auth = globus_sdk.NativeAppAuthClient(CLIENT_ID)
auth.oauth2_start_flow(requested_scopes=scope)
print("Sign in here:\n", auth.oauth2_get_authorize_url())
code = input("Authorization code: ").strip()
tok = auth.oauth2_exchange_code_for_tokens(code).by_resource_server["transfer.api.globus.org"]
tc = globus_sdk.TransferClient(authorizer=globus_sdk.AccessTokenAuthorizer(tok["access_token"]))
print("signed in")

In [ ]:
print(f"Cape Town collection {CAPE_COLLECTION}, {CAPE_SRC_PATH}:")
for e in tc.operation_ls(CAPE_COLLECTION, path=CAPE_SRC_PATH):
    print(f"  {e['type']:4s} {e['size']:>15,d}  {e['name']}")

## 5. Transfers, watched from the DTN's sockets

During each transfer, the DTN's open TCP connections on the GridFTP data ports are read every few seconds. Each
connection is assigned to the interface it uses: the **data NIC** (green, the Cape Town path) or the **public
uplink** (red). The chart shows Gb/s per path live, and the card below it gives the verdict.

In [ ]:
SS = ("ss -Htni state established "
      "'( sport >= :50000 and sport <= :51000 ) or ( dport >= :50000 and dport <= :51000 )'")

def _ip(ep):
    return ep.rsplit(":", 1)[0].strip("[]").replace("::ffff:", "")

def data_conns(n):
    # (local, remote) -> bytes carried so far, for every GridFTP data connection right now
    conns, cur = {}, None
    for line in n.execute(SS, quiet=True)[0].splitlines():
        if not line.strip():
            continue
        if not line[0].isspace():
            f = line.split()
            cur = (f[2], f[3]) if len(f) >= 4 else None
            if cur: conns[cur] = 0
        elif cur:
            got = [int(x) for x in re.findall(r"bytes_(?:received|acked):(\d+)", line)]
            conns[cur] = max(got or [0])
    return conns

def path_of(l, local_ip):
    if local_ip == l["data_ip"]:
        return "data NIC (Cape Town path)"
    if local_ip in (l.get("public_ip"), l.get("public_ip6")):
        return "public uplink"
    return "other"

DONE = ("SUCCEEDED", "FAILED")

def watch_task(site, task_id, title, every=3):
    l, n = legs[site], node(site)
    plot = demo_viz.LivePathPlot(f"{site}: {title}")
    seen = {}
    while True:
        for k, b in data_conns(n).items():
            seen[k] = max(seen.get(k, 0), b)
        totals = {}
        for (loc, _), b in seen.items():
            p = path_of(l, _ip(loc)); totals[p] = totals.get(p, 0) + b
        t = tc.get_task(task_id)
        plot.update(totals, t["status"] + (f" ({t['nice_status']})" if t.get("nice_status") else ""),
                    t["bytes_transferred"])
        if t["status"] in DONE or t.get("nice_status") in ("PERMISSION_DENIED", "ENDPOINT_ERROR"):
            break
        time.sleep(every)
    card = demo_viz.path_summary(site, l, totals, t); display(card); plt.close(card)
    for (loc, rem), b in sorted(seen.items()):
        print(f"  {_ip(loc):>15s} <- {_ip(rem):15s} {b:>15,d} B  {path_of(l, _ip(loc))}")
    if not seen:
        print("  no data connection was caught - use a larger dataset (or a smaller `every`).")
    return t

def transfer_and_watch(site, src, src_path, dst, dst_path, label, recursive=False):
    td = globus_sdk.TransferData(source_endpoint=src, destination_endpoint=dst, label=label,
                                 verify_checksum=True)
    td.add_item(src_path, dst_path, recursive=recursive)
    task = tc.submit_transfer(td)["task_id"]
    print(f"task {task}: {src_path} -> {dst_path}")
    return watch_task(site, task, label)

### 5a. Cape Town → FABRIC

In [ ]:
for site, l in legs.items():
    if l.get("globus") != "ready":
        continue
    acct = next(iter(l["users"].values()))           # first mapped account; its private directory
    is_dir = CAPE_SRC_PATH.endswith("/")
    transfer_and_watch(site, CAPE_COLLECTION, CAPE_SRC_PATH, l["collection_id"],
                       f"/{acct}/from-cape/" + ("" if is_dir else os.path.basename(CAPE_SRC_PATH)),
                       f"Cape Town to {site} (SC26)", recursive=is_dir)

### 5b. FABRIC → Cape Town

This needs write access on the Cape Town collection. It generates a `PUSH_SIZE_GB` file in the mapped account's
directory on the DTN and pushes it to `CAPE_DST_DIR`.

In [ ]:
try:
    tc.operation_mkdir(CAPE_COLLECTION, CAPE_DST_DIR)
except globus_sdk.TransferAPIError as e:
    if "Exists" not in str(e.code):
        raise
for site, l in legs.items():
    if l.get("globus") != "ready":
        continue
    acct = next(iter(l["users"].values()))
    fname = f"sc26-{site.lower()}-{PUSH_SIZE_GB}G.bin"
    node(site).execute(f"sudo -u {acct} bash -c 'f={l['data_base']}/{acct}/{fname}; "
                       f"[ -s $f ] || head -c {PUSH_SIZE_GB}G /dev/urandom > $f'", quiet=True)
    transfer_and_watch(site, l["collection_id"], f"/{acct}/{fname}", CAPE_COLLECTION,
                       CAPE_DST_DIR + fname, f"{site} to Cape Town (SC26)")

### 5c. Start it from the Globus web app: same path

This shows the audience that the route doesn't depend on where the transfer is started.
1. Run the next cell. It prints a web-app link with both collections already open.
2. In the browser, select files on the Cape Town side and click **Start**.
3. The cell picks up the new task by itself and charts it exactly as above. The green line should look the same.

In [ ]:
SITE = next(s for s, l in legs.items() if l.get("globus") == "ready")
l = legs[SITE]
print(f"Open: https://app.globus.org/file-manager?origin_id={CAPE_COLLECTION}"
      f"&destination_id={l['collection_id']}&destination_path=/{next(iter(l['users'].values()))}/")
since = datetime.now(timezone.utc)
print("Waiting for a transfer started in the web app (5 min)...")

PAIR = {CAPE_COLLECTION, l["collection_id"]}   # only a transfer between Cape Town and THIS DTN counts

def new_task():
    """Newest transfer started after `since` whose two ends are exactly Cape Town and this
    site's collection (either direction) - nothing else on the account is picked up."""
    hits = []
    for t in tc.task_list(limit=50, filter="type:TRANSFER/status:ACTIVE,SUCCEEDED"):
        started = datetime.fromisoformat(t["request_time"].replace("Z", "+00:00"))
        if started >= since and {t["source_endpoint_id"], t["destination_endpoint_id"]} == PAIR:
            hits.append((started, t["task_id"]))
    return max(hits)[1] if hits else None

for _ in range(100):
    task_id = new_task()
    if task_id:
        print(f"found task {task_id} started from the web app")
        watch_task(SITE, task_id, "started from the Globus web app")
        break
    time.sleep(3)
else:
    print("no web-app transfer seen - start one and re-run this cell")

## 6. Same thing from the Globus CLI

Same command, same path:

In [ ]:
for site, l in legs.items():
    if l.get("collection_id"):
        acct = next(iter(l["users"].values()))
        print(f"# {site}")
        print(f"globus transfer {CAPE_COLLECTION}:{CAPE_SRC_PATH} {l['collection_id']}:/{acct}/from-cape/ "
              f"--recursive --verify-checksum --label 'Cape Town to {site}'\n")

## 7. After the demo: unregister the endpoint (optional)

Deletes each DTN's Globus endpoint and its collection, the same steps the weave's stop runs first: it detaches
the node (`node cleanup`), then deletes the endpoint with the deployment key kept on the DTN (`endpoint cleanup`).
The slice is left alone; delete it afterwards with `ACTION=stop ./weave.sh <prefix>`, which then finds nothing
left to unregister.

**This can't be undone.** A new run of the weave registers a new endpoint, with a new collection ID and link.
Set `UNREGISTER = True` to do it.

In [ ]:
import provision

UNREGISTER = False   # True = delete the Globus endpoint(s) and collection(s); the slices stay

for site, l in legs.items():
    if not UNREGISTER:
        print(f"{site}: endpoint {l.get('collection_id') and 'registered' or 'not registered'} - "
              "set UNREGISTER = True to delete it")
        continue
    try:
        provision.remove_endpoint(node(site), site)
        print(f"{site}: endpoint removed (or was never registered). Slice {l['slice']} is still up.")
    except Exception as e:
        print(f"{site}: FAILED - {e}\n  The slice is kept so this can be retried from the DTN.")

## Recap

- **What moved:** the cards above give the GB, the effective Gb/s, and the path each byte took.
- **Why it took the research VLAN:** the FABRIC endpoint advertises its VLAN address as its data interface, and the Cape Town side connects to it. That is one setting at registration time, `--data-interface`.
- **What it didn't depend on:** CLI, notebook or web app, all three started transfers that took the same green path.
- **How it was built:** one FABRIC weave (`cape-globus-l2`) creates the VM and bridges it onto the AmLight VLAN through a facility port. It gives the VM a public AmLight address for Globus, installs and firewalls GCS, and registers the endpoint. The same weave builds FIU and CIEN.